<a href="https://colab.research.google.com/github/suhailabdulwasi/ds2002-fa26/blob/main/06-midterm-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [9]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [10]:
pop_rows = cat[cat["product_name"].str.contains("pop", case=False, na=False)]

print("Count:", len(pop_rows))
print(pop_rows)

Count: 3
             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [11]:

strawberry = pop_rows[
    pop_rows["product_name"].str.contains("strawberry", case=False, na=False)
]

print(strawberry[["sku", "product_name"]])

print("Other Pop-Tart products:")
print(pop_rows[~pop_rows.index.isin(strawberry.index)])


             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
Other Pop-Tart products:
Empty DataFrame
Columns: [sku, product_name, category]
Index: []


Pop-Tart rows that are a different product: None. All three Pop-Tart rows represent strawberry Pop-Tarts.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [18]:
original_count = len(cat)
sku_map = {
    "PT-12": "PT-12",
    "SKU#459812": "PT-12",
    "POPTART-STRAW": "PT-12"
}

cat["canonical_sku"] = cat["sku"].replace(sku_map)

print(cat[["sku", "product_name", "canonical_sku"]])


             sku          product_name canonical_sku
0          PT-12  pop-tarts strawberry         PT-12
1     SKU#459812   Strawberry Pop-Tart         PT-12
2  POPTART-STRAW  Pop-Tarts Strawberry         PT-12


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [13]:
print("Before:")
print(cat["category"].value_counts())

cat["category"] = cat["category"].str.strip().str.title()

print("\nAfter:")
print(cat["category"].value_counts())

Before:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

After:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [14]:
import re

def normalize_name(name):
    name = name.lower()
    name = re.sub(r"[^\w\s]", " ", name)
    return " ".join(sorted(name.split()))

cat["normalized_name"] = cat["product_name"].apply(normalize_name)

groups = cat.groupby("normalized_name").filter(
    lambda x: x["sku"].nunique() > 1
)

print(groups[["sku", "product_name", "normalized_name"]])


             sku          product_name       normalized_name
0          PT-12  pop-tarts strawberry  pop strawberry tarts
2  POPTART-STRAW  Pop-Tarts Strawberry  pop strawberry tarts


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [19]:
assert cat["canonical_sku"].isna().sum() == 0
assert len(cat) == original_count

print("Mapping is safe!")
print("No null SKUs and row count is unchanged.")

Mapping is safe!
No null SKUs and row count is unchanged.


### Write-up

a) Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

In this case, there were only duplicates for strawberry Pop-Tarts in the database. For other product types, I would use name and size comparison to consolidate SKUs.

b) Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

Consolidation is a business decision when similar SKUs could possibly differ. I would approach the catalog manager to find out whether SKUs that vary in packaging size can be consolidated.